# ARC-AGI-2: two-pass grid adaptation with verified geometry

Version5 preserves version3's model, seven core solver files, two64-sequence/8-view adaptations, seeds, budgets, probability selector and first-guess anchor. It adds only a bounded CPU geometry pass: border-ray/obstacle propagation and masked-reflection reconstruction. A rule must exactly fit every demonstration, and all admitted rules must agree on its test predictions. Only the second guess may change; no task-ID lookup or evaluation answers enter inference.

The geometric code adapts Ayodeji Ibrahim Lateef's [Apache-2.0 ARC2 v54 notebook](https://www.kaggle.com/code/ayodejiibrahimlateef/arc2-v54-verified-geometry-hybrid-l4), version2(345975616), with its original symbolic/rule foundations credited there. The unchanged neural solution credits Luxshan Thavarasa's Apache-2.0 NVARC notebook, NVARC, Ivan Sorokin, Jean-François Puget, mikelou1, the ARChitects and koushikrudra. The Sorokin Qwen3 grid checkpoint is unchanged.

On saved version3 predictions, this fixed geometry policy added one correct released-evaluation output with no losses:36.11%/57exact versus35.28%/56. That is development evidence using a public, released-task-informed solver, not independent confirmation or public-score improvement. A fresh preview remains to be measured. Best verified public score remains29.72%; no rank gain is promised.

Preview runs all120releasedtasks under6h; competition reruns read current hidden inputs under12h with reporting reserve. Geometry has a120-second process limit. No cached reference predictions are included, and fixed predictions are saved before released labels. Previous symbolic policies showed no incremental gain; this newly audited policy is distinct.

In [ ]:
import os,time,json,hashlib
from pathlib import Path
T0=time.time()
RERUN=os.getenv('KAGGLE_IS_COMPETITION_RERUN','').lower() in ('1','true','yes')
budget_seconds=12*3600 if RERUN else 6*3600
global_end_time=T0+budget_seconds-600
data_roots=[Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2'),Path('/kaggle/input/arc-prize-2026-arc-agi-2')]
file_name='arc-agi_test_challenges.json' if RERUN else 'arc-agi_evaluation_challenges.json'
data_root=next(p for p in data_roots if (p/file_name).is_file())
input_path=data_root/('arc-agi_test_challenges.json' if RERUN else 'arc-agi_evaluation_challenges.json')
tasks=json.loads(input_path.read_text())
assert all('output' not in pair for task in tasks.values() for pair in task['test'])
run_config={'input_path':str(input_path),'deadline':global_end_time,'mode':'hidden' if RERUN else 'released_evaluation','nominal_budget_seconds':budget_seconds}
Path('run_config.json').write_text(json.dumps(run_config,indent=2))
print('Mode',run_config['mode'],'tasks',len(tasks),'outputs',sum(len(t['test']) for t in tasks.values()))


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','tensorflow'],check=True)
import torch
assert torch.cuda.device_count()==4
print('CUDA',torch.version.cuda,'devices',[torch.cuda.get_device_name(i) for i in range(4)])


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


def _valid_sample(sample):
    try:
        sol = np.asarray(sample["solution"])
        if sol.ndim != 2 or sol.size == 0 or sol.shape[0] > 30 or sol.shape[1] > 30:
            return False
        if not np.issubdtype(sol.dtype, np.integer):
            return False
        if sol.min() < 0 or sol.max() > 9:
            return False
        if not np.isfinite(sample["beam_score"]):
            return False
        if not len(sample["score_aug"]) or not np.all(np.isfinite(sample["score_aug"])):
            return False
        return True
    except Exception:
        return False


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        try:
            self.valid_keys = set(dataset.keys)
        except Exception:
            self.valid_keys = None

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"*** No decoded results at {store}")
            return 0
        n_files = n_samples = n_bad = 0
        for key in sorted(os.listdir(store)):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping corrupt shard {key}: {e}")
                n_bad += 1
                continue
            n_files += 1
            base_key = key.split(".")[0]
            if self.valid_keys is not None and base_key not in self.valid_keys:
                n_bad += 1
                continue
            for i, sample in enumerate(outputs):
                if not _valid_sample(sample):
                    n_bad += 1
                    continue
                self.decoded_results.setdefault(base_key, {})[f"{key}{run_name}.out{i}"] = sample
                n_samples += 1
        print(f"*** Loaded {n_files} shards / {n_samples} samples from {store} (skipped {n_bad})")
        return n_samples

    def candidate_stats(self):
        """Per output key: number of unique candidate grids, number of samples, and the
        vote count / kgmon margin of the leading candidate (how sure the pool is)."""
        stats = {}
        for bk, v in self.decoded_results.items():
            groups = {}
            for g in v.values():
                groups.setdefault(hashable(g["solution"]), []).append(g)
            scored = sorted((getter_kgmon(gs) for gs in groups.values()), reverse=True)
            top_votes = max(len(gs) for gs in groups.values())
            margin = (scored[0] - scored[1]) if len(scored) > 1 else float("inf")
            stats[bk] = dict(unique=len(groups), samples=len(v), top_votes=top_votes, margin=margin)
        return stats

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import zlib
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle
import traceback

logging.disable(logging.WARNING)

# ---------------------------------------------------------------------------
# Run configuration (all overridable through environment variables so the
# same solver can run the LB-proven primary pass and a wider secondary pass).
# Defaults == the public LB 33.89 NVARC configuration.
# ---------------------------------------------------------------------------
def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default

CFG = dict(
    model_path      = _env("ARC_MODEL_PATH", "", str),
    out_dir         = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    lora_seed       = _env("ARC_LORA_SEED", 42, int),
    train_aug_seed  = _env("ARC_TRAIN_AUG_SEED", 1, int),
    n_train_aug     = _env("ARC_N_TRAIN_AUG", 16, int),     # x8 geometries = 128 sequences
    num_epochs      = _env("ARC_EPOCHS", 1, int),
    learning_rate   = _env("ARC_LR", 5e-5, float),
    eval_aug_seed   = _env("ARC_EVAL_AUG_SEED", 2, int),
    n_eval_aug      = _env("ARC_N_EVAL_AUG", 2, int),       # x8 geometries = 16 decoded views
    min_prob        = _env("ARC_MIN_PROB", 0.2, float),     # DFS cumulative prob threshold
    dfs_window      = _env("ARC_DFS_WINDOW", 540.0, float), # seconds per DFS call
    task_cap        = _env("ARC_TASK_CAP", 1200.0, float),  # seconds per task (decode phase)
    score_seed_off  = _env("ARC_SCORE_SEED_OFFSET", 0, int),
    decode_batch    = _env("ARC_DECODE_BATCH", 4, int),
)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def resolve_model_dir():
    if CFG["model_path"] and os.path.isdir(CFG["model_path"]):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    import glob
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if "grids15" in d and os.path.isfile(os.path.join(d, "tokenizer.json")):
            return d
    raise FileNotFoundError("qwen3_4b_grids15_sft139 model directory not found under /kaggle/input")


def stable_seed(key, offset=0):
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Throughput patch (algebraically identical to the baseline): only the 12
# ARC-token NLL values are moved to the CPU instead of the whole vocabulary.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_window) -> dict:

    n = logits.size(0)

    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < dfs_window and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_window=dfs_window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        dfs_window=dfs_window,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def make_view_batches(eval_ds, n_perm, batch_size):
    """Group decoded views into batches. With n_perm=2 / batch_size=4 this is
    byte-for-byte the baseline batching ([id,rot180],[rot90,rot270],[T,T.rot180],[T.rot90,T.rot270])."""
    test_id_to_subkeys = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        test_id_to_subkeys[test_id].append(subkey)
    # sorted geometry order: 0 id, 1 rot90, 2 rot180, 3 rot270, 4 T, 5 T.rot90, 6 T.rot180, 7 T.rot270
    groups_a = [0, 2, 1, 3]          # untransposed geometries
    groups_b = [4, 6, 5, 7]          # transposed geometries
    batches = []
    for geos in (groups_a, groups_b):
        for test_id, subkeys in test_id_to_subkeys.items():
            if n_perm == 2 and batch_size == 4:
                # exact baseline batches: [id,rot180], [rot90,rot270], ...
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    batches.append(subkeys[a*n_perm:(a+1)*n_perm] + subkeys[b*n_perm:(b+1)*n_perm])
            else:
                # Batched DFS needs equal token lengths in a batch. Within a geometry group the
                # shape classes are: rot0/rot180 = H x W, rot90/rot270 = W x H, so pair them.
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    views = subkeys[a*n_perm:(a+1)*n_perm] + subkeys[b*n_perm:(b+1)*n_perm]
                    for i in range(0, len(views), batch_size):
                        batches.append(views[i:i+batch_size])
    return batches


def worker(rank, queue, end_time, test_path=None):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=CFG["lora_seed"],
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=CFG["num_epochs"],
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=CFG["learning_rate"],
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=CFG["lora_seed"],
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model_dir = resolve_model_dir()
    print(f"[Rank {rank}] model dir: {model_dir}")
    print(f"[Rank {rank}] config: {CFG}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_dir,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(CFG["min_prob"])

    if test_path is None:
        if rerun_mode:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = CFG["out_dir"]
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()

        try:
            torch.cuda.reset_peak_memory_stats()

            set_peft_model_state_dict(
                model,
                default_weights.copy(),
                adapter_name="default",
            )

            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])

            train_ds = puzzle_ds.augment(n=CFG["n_train_aug"], shfl_keys=True, seed=CFG["train_aug_seed"])
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

            model = FastLanguageModel.for_inference(model)

            gc.collect()
            torch.cuda.empty_cache()

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

            torch.cuda.reset_peak_memory_stats()

            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_ds = puzzle_ds_multi.augment(n=CFG["n_eval_aug"], seed=CFG["eval_aug_seed"])
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            batches = make_view_batches(eval_ds, CFG["n_eval_aug"], CFG["decode_batch"])

            with torch.inference_mode():

                known_scores = {}

                for subkeys in batches:

                    spend_time = time.time() - start_time
                    if spend_time > CFG["task_cap"] or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time, CFG["dfs_window"])

                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens_ in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens_)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed(bk, CFG["score_seed_off"]))
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores

                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")

        except Exception as e:
            print(f"[Rank {rank}] ERROR on puzzle {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            try:
                model = FastLanguageModel.for_inference(model)
            except Exception:
                pass
            gc.collect()
            torch.cuda.empty_cache()
            if isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.synchronize()

        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import sys
import time
import json
import torch
import argparse
import traceback
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, test_path, marker_dir):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Serialize the unsloth import/patching across workers (baseline behaviour),
    # but never wait forever for a dead predecessor.
    if rank > 0:
        waited = 0
        while not os.path.exists(os.path.join(marker_dir, f"worker{rank-1}")) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    attempts = 0
    while attempts < 2 and time.time() < end_time:
        attempts += 1
        try:
            worker(rank, queue, end_time, test_path=test_path)
            break
        except Exception as e:
            print(f"[Rank {rank}] worker crashed ({type(e).__name__}: {e}); attempt {attempts}")
            traceback.print_exc()
            try:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            if attempts >= 2:
                print(f"[Rank {rank}] giving up.")

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    """Rough cost proxy: tokens of all train pairs (TTT + context) plus an estimate of
    the decoded output sizes. Cheap tasks first => the deadline truncates the most
    expensive (and least likely to be solved) tail."""
    def ntok(g):
        return len(g) * (len(g[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--nprocs", type=int, default=0)
    parser.add_argument("--order", type=str, default="cheap", choices=["cheap", "sorted", "file"])
    parser.add_argument("--test-path", type=str, default="")
    parser.add_argument("--marker-dir", type=str, default="/kaggle/working/markers")
    args, _ = parser.parse_known_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if args.test_path:
        test_path = args.test_path
    elif rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    else:
        keys = sorted(data.keys())
        if not rerun_mode:
            debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
            keys = [k for k in keys if k in debug_keys]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "sorted":
        keys = sorted(keys)

    nprocs = args.nprocs or min(4, torch.cuda.device_count())
    os.makedirs(args.marker_dir, exist_ok=True)
    for f_ in os.listdir(args.marker_dir):
        try:
            os.remove(os.path.join(args.marker_dir, f_))
        except Exception:
            pass

    print(f"[starter] {len(keys)} tasks, {nprocs} workers, order={args.order}, "
          f"budget={(args.end_time - time.time())/60:.1f} min, test_path={test_path}")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    try:
        mp.spawn(local_worker, args=(queue, args.end_time, test_path, args.marker_dir), nprocs=nprocs)
    except Exception as e:
        print(f"[starter] spawn finished with error: {type(e).__name__}: {e}")
        traceback.print_exc()
    print("[starter] finished.")


In [ ]:
%%writefile reasoning_study_policy.py
"""Grid validation and fixed policies for independent reasoning samples."""
import copy


def valid_grid(grid):
    return (isinstance(grid, list) and 0 < len(grid) <= 30
            and isinstance(grid[0], list) and 0 < len(grid[0]) <= 30
            and all(isinstance(row, list) and len(row) == len(grid[0])
                    and all(type(x) is int and 0 <= x <= 9 for x in row)
                    for row in grid))


def valid_answers(grids, task):
    return isinstance(grids, list) and len(grids) == len(task['test']) and all(map(valid_grid, grids))


def validate_submission(prediction, tasks):
    assert set(prediction) == set(tasks)
    for key, task in tasks.items():
        assert len(prediction[key]) == len(task['test'])
        for row in prediction[key]:
            assert set(row) == {'attempt_1', 'attempt_2'}
            assert all(map(valid_grid, row.values()))


def candidate_samples(tasks, records):
    samples = {key: [] for key in tasks}
    seen = set()
    for record in sorted(records, key=lambda r: (r['task_id'], r['sample'])):
        key = record['task_id']
        identity = (key, record['sample'])
        assert key in tasks and identity not in seen, identity
        seen.add(identity)
        if valid_answers(record.get('grids'), tasks[key]):
            samples[key].append(record['grids'])
    return samples


def model_submission(tasks, records):
    samples = candidate_samples(tasks, records)
    submission = {}
    for key, task in tasks.items():
        rows = []
        for index in range(len(task['test'])):
            candidates = []
            for grids in samples[key]:
                if grids[index] not in candidates:
                    candidates.append(grids[index])
            rows.append({'attempt_1': candidates[0] if candidates else [[0]],
                         'attempt_2': candidates[1] if len(candidates) > 1 else (candidates[0] if candidates else [[0]])})
        submission[key] = rows
    validate_submission(submission, tasks)
    return submission


def fixed_policies(tasks, baseline, records):
    validate_submission(baseline, tasks)
    samples = candidate_samples(tasks, records)
    policies = {name: copy.deepcopy(baseline) for name in
                ('baseline', 'replace_second', 'agreement_second', 'duplicate_only')}
    policies['reasoning_only'] = model_submission(tasks, records)
    for key, task in tasks.items():
        for index in range(len(task['test'])):
            first = baseline[key][index]['attempt_1']
            candidates = [grids[index] for grids in samples[key]]
            candidate = next((g for g in candidates if g != first), None)
            if candidate is None:
                continue
            policies['replace_second'][key][index]['attempt_2'] = candidate
            if sum(g == candidate for g in candidates) >= 2:
                policies['agreement_second'][key][index]['attempt_2'] = candidate
            if baseline[key][index]['attempt_2'] in (first, [[0]]):
                policies['duplicate_only'][key][index]['attempt_2'] = candidate
    for prediction in policies.values():
        validate_submission(prediction, tasks)
    return policies


In [ ]:
%%writefile two_pass_policy.py
"""Pool independent grid-model passes while retaining primary top-1."""
from reasoning_study_policy import valid_grid


def anchored(primary, pooled):
    selected = {}
    for key in sorted(set(primary) | set(pooled)):
        candidates = primary.get(key, [])[:1] + pooled.get(key, [])
        unique = []
        for grid in candidates:
            if grid not in unique:
                unique.append(grid)
        selected[key] = unique[:2]
    return selected


def submission_for(tasks, ranked):
    prediction = {}
    for key, task in tasks.items():
        prediction[key] = []
        for index in range(len(task['test'])):
            candidates = ranked.get(f'{key}_{index}', [])[:2]
            assert all(valid_grid(g) for g in candidates), (key, index)
            prediction[key].append({'attempt_1': candidates[0] if candidates else [[0]],
                                    'attempt_2': candidates[1] if len(candidates) > 1 else [[0]]})
    return prediction


In [ ]:
%%writefile two_pass_run.py
"""Run two bounded independent adaptation passes on the current input file."""
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time


def phase(name, keys, deadline, environment):
    Path(f'keys_{name}.json').write_text(json.dumps(keys))
    started = time.time()
    if deadline-started < 60:
        return {'phase': name, 'tasks': len(keys), 'skipped': True}
    command = [sys.executable, 'starter.py', '--end-time', str(deadline),
               '--keys-file', f'keys_{name}.json', '--order', 'file',
               '--test-path', environment['ARC_INPUT_PATH']]
    with open(f'pass_{name}.log', 'w') as logfile:
        process = subprocess.Popen(command, env=environment, stdout=logfile, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        stopped = False
        try:
            code = process.wait(timeout=max(1, deadline-time.time()))
        except subprocess.TimeoutExpired:
            stopped = True
            os.killpg(process.pid, signal.SIGTERM)
            try:
                code = process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                code = process.wait()
    result = {'phase': name, 'tasks': len(keys), 'seconds': time.time()-started,
              'returncode': code, 'hard_stopped_at_deadline': stopped}
    Path(f'phase_{name}.json').write_text(json.dumps(result, indent=2))
    if code != 0 and not stopped:
        raise RuntimeError(f'Phase {name} failed: inspect pass_{name}.log')
    return result


def main(configuration):
    from starter import estimated_work
    tasks = json.loads(Path(configuration['input_path']).read_text())
    assert all('output' not in item for task in tasks.values() for item in task['test'])
    keys = sorted(tasks, key=lambda k: (estimated_work(tasks[k]), k))
    common = dict(os.environ, UNSLOTH_DISABLE_STATISTICS='1', TRITON_PTXAS_PATH='/usr/local/cuda/bin/ptxas',
                  OMP_NUM_THREADS='12', PYTHONHASHSEED='0', ARC_N_TRAIN_AUG='8', ARC_N_EVAL_AUG='1',
                  ARC_INPUT_PATH=configuration['input_path'], ARC_TASK_CAP='800')
    started = time.time()
    midpoint = started+(configuration['deadline']-started)*.5
    a = phase('a', keys, midpoint, dict(common, ARC_OUT_DIR='/kaggle/inference_outputs',
                                      ARC_LORA_SEED='42', ARC_TRAIN_AUG_SEED='1', ARC_EVAL_AUG_SEED='2', ARC_SCORE_SEED_OFFSET='0'))
    covered = {p.name.split('.')[0].rsplit('_', 1)[0] for p in Path('/kaggle/inference_outputs').glob('*')}
    order_b = [k for k in keys if k not in covered]+[k for k in keys if k in covered]
    b = phase('b', order_b, configuration['deadline'], dict(common, ARC_OUT_DIR='/kaggle/inference_outputs_b',
                      ARC_LORA_SEED='137', ARC_TRAIN_AUG_SEED='17', ARC_EVAL_AUG_SEED='29', ARC_SCORE_SEED_OFFSET='7'))
    Path('two_pass_runtime.json').write_text(json.dumps({'configuration': configuration, 'phases': [a, b],
                                                       'total_seconds': time.time()-started}, indent=2))


if __name__ == '__main__':
    main(json.loads(sys.argv[1]))


In [ ]:
%%writefile verified_geometry.py
# Derived from Ayodeji Ibrahim Lateef's ARC2 v54 Verified Geometry Hybrid L4.
# Source version2,345975616; Apache-2.0. Only the exact geometry path is retained.
# https://www.kaggle.com/code/ayodejiibrahimlateef/arc2-v54-verified-geometry-hybrid-l4
# See LICENSE.geometry.txt and notebook attribution.
from __future__ import annotations
from dataclasses import dataclass
from typing import List,Tuple
import numpy as np
Grid=Tuple[Tuple[int,...],...]

def to_arr(grid) -> np.ndarray:
    return np.asarray(grid, dtype=np.int8)

def to_grid(arr: np.ndarray) -> Grid:
    arr = np.asarray(arr, dtype=np.int8)
    return tuple((tuple((int(x) for x in row)) for row in arr.tolist()))

@dataclass
class Candidate:
    name: str
    outputs: List[np.ndarray]
    score: float

class Arc2SymbolicSolver:

    def _pairs(self, task: dict):
        return [(to_arr(p['input']), to_arr(p['output'])) for p in task['train']]

    def _simulate_border_rays(self, source, background, ray_color, obstacle_colors):
        grid = source.copy()
        height, width = grid.shape
        active = {col for col, value in enumerate(source[0]) if int(value) == int(ray_color)}
        if not active:
            return None
        collisions = 0
        for row in range(1, height):
            runs = []
            col = 0
            while col < width:
                if int(source[row, col]) not in obstacle_colors:
                    col += 1
                    continue
                left = col
                while col + 1 < width and int(source[row, col + 1]) in obstacle_colors:
                    col += 1
                runs.append((left, col))
                col += 1
            blocked = set()
            spawned = set()
            for left, right in runs:
                hit = {col for col in active if left <= col <= right}
                if not hit:
                    continue
                collisions += 1
                blocked.update(hit)
                for bypass_col in range(max(0, left - 1), min(width - 1, right + 1) + 1):
                    if int(grid[row - 1, bypass_col]) == int(background):
                        grid[row - 1, bypass_col] = ray_color
                for bypass_col in (left - 1, right + 1):
                    if 0 <= bypass_col < width:
                        spawned.add(bypass_col)
                        if int(grid[row, bypass_col]) == int(background):
                            grid[row, bypass_col] = ray_color
            active = active - blocked | spawned
            for active_col in active:
                if int(grid[row, active_col]) == int(background):
                    grid[row, active_col] = ray_color
        if collisions == 0:
            return None
        return grid

    def _fit_border_ray_obstacle(self, task):
        pairs = self._pairs(task)
        tests = [to_arr(item['input']) for item in task['test']]
        if not pairs or any((source.shape != target.shape for source, target in pairs)):
            return
        transitions = {(int(source_value), int(target_value)) for source, target in pairs for source_value, target_value in zip(source.ravel(), target.ravel()) if int(source_value) != int(target_value)}
        if len(transitions) != 1:
            return
        background, ray_color = next(iter(transitions))
        if background == ray_color:
            return
        for source, target in pairs:
            allowed_change = (source == background) & (target == ray_color)
            if np.any((source != target) & ~allowed_change):
                return
        all_colors = {int(value) for source, _target in pairs for value in np.unique(source)}
        obstacle_colors = all_colors - {background, ray_color}
        if not obstacle_colors:
            return
        for rotation in range(4):
            valid = True
            for source, target in pairs:
                normalized = np.rot90(source, rotation)
                prediction = self._simulate_border_rays(normalized, background, ray_color, obstacle_colors)
                if prediction is None:
                    valid = False
                    break
                prediction = np.rot90(prediction, -rotation)
                if not np.array_equal(prediction, target):
                    valid = False
                    break
            if not valid:
                continue
            outputs = []
            for test in tests:
                prediction = self._simulate_border_rays(np.rot90(test, rotation), background, ray_color, obstacle_colors)
                if prediction is None:
                    valid = False
                    break
                outputs.append(np.rot90(prediction, -rotation))
            if valid:
                yield Candidate(f'verified_border_rays:r{rotation}', outputs, 1.2)

    def _solid_color_rectangle(self, source, color):
        rows, cols = np.where(source == color)
        if len(rows) == 0:
            return None
        top = int(rows.min())
        bottom = int(rows.max()) + 1
        left = int(cols.min())
        right = int(cols.max()) + 1
        if (bottom - top) * (right - left) != len(rows):
            return None
        if not np.all(source[top:bottom, left:right] == color):
            return None
        return (top, bottom, left, right)

    def _recover_masked_reflection(self, source, mask_color):
        rectangle = self._solid_color_rectangle(source, mask_color)
        if rectangle is None:
            return None
        top, bottom, left, right = rectangle
        output_height = bottom - top
        output_width = right - left
        height, width = source.shape
        non_mask_count = int(np.count_nonzero(source != mask_color))
        minimum_support = max(20, (non_mask_count + 1) // 2)
        reflection_maps = []
        for orientation, size in (('horizontal', height), ('vertical', width)):
            for doubled_axis in range(2 * size - 1):
                matching = 0
                support = 0
                if orientation == 'horizontal':
                    for row in range(height):
                        reflected_row = doubled_axis - row
                        if not 0 <= reflected_row < height:
                            continue
                        valid = (source[row, :] != mask_color) & (source[reflected_row, :] != mask_color)
                        support += int(np.count_nonzero(valid))
                        matching += int(np.count_nonzero((source[row, :] == source[reflected_row, :]) & valid))
                else:
                    for col in range(width):
                        reflected_col = doubled_axis - col
                        if not 0 <= reflected_col < width:
                            continue
                        valid = (source[:, col] != mask_color) & (source[:, reflected_col] != mask_color)
                        support += int(np.count_nonzero(valid))
                        matching += int(np.count_nonzero((source[:, col] == source[:, reflected_col]) & valid))
                if support >= minimum_support and matching == support:
                    reflection_maps.append((orientation, doubled_axis))
        if not reflection_maps:
            return None
        partial = np.full((output_height, output_width), -1, dtype=np.int16)
        for out_row in range(output_height):
            for out_col in range(output_width):
                row = top + out_row
                col = left + out_col
                reflected_values = []
                for orientation, doubled_axis in reflection_maps:
                    if orientation == 'horizontal':
                        source_row = doubled_axis - row
                        source_col = col
                    else:
                        source_row = row
                        source_col = doubled_axis - col
                    if 0 <= source_row < height and 0 <= source_col < width and (int(source[source_row, source_col]) != int(mask_color)):
                        reflected_values.append(int(source[source_row, source_col]))
                if reflected_values:
                    if len(set(reflected_values)) != 1:
                        return None
                    partial[out_row, out_col] = reflected_values[0]
        known = partial >= 0
        if int(np.count_nonzero(known)) < max(4, (output_height * output_width + 2) // 3):
            return None
        candidates = {}
        source_shapes = {(output_height, output_width), (output_width, output_height)}
        for patch_height, patch_width in source_shapes:
            if patch_height > height or patch_width > width:
                continue
            for row in range(height - patch_height + 1):
                for col in range(width - patch_width + 1):
                    patch = source[row:row + patch_height, col:col + patch_width]
                    if np.any(patch == mask_color):
                        continue
                    variants = (patch, np.rot90(patch, 1), np.rot90(patch, 2), np.rot90(patch, 3), np.flipud(patch), np.fliplr(patch), patch.T, np.flipud(np.fliplr(patch)).T)
                    for variant in variants:
                        if variant.shape != partial.shape:
                            continue
                        if not np.array_equal(variant[known], partial[known]):
                            continue
                        key = to_grid(variant)
                        candidates[key] = variant.copy()
        if len(candidates) != 1:
            return None
        return next(iter(candidates.values()))

    def _fit_masked_reflection(self, task):
        pairs = self._pairs(task)
        tests = [to_arr(item['input']) for item in task['test']]
        if not pairs:
            return
        shared_mask_colors = None
        for source, target in pairs:
            output_colors = {int(value) for value in np.unique(target)}
            pair_mask_colors = set()
            for value in np.unique(source):
                color = int(value)
                if color in output_colors:
                    continue
                rectangle = self._solid_color_rectangle(source, color)
                if rectangle is None:
                    continue
                top, bottom, left, right = rectangle
                if (bottom - top, right - left) == target.shape:
                    pair_mask_colors.add(color)
            if shared_mask_colors is None:
                shared_mask_colors = pair_mask_colors
            else:
                shared_mask_colors &= pair_mask_colors
        if not shared_mask_colors:
            return
        for mask_color in sorted(shared_mask_colors):
            if not all(((prediction := self._recover_masked_reflection(source, mask_color)) is not None and np.array_equal(prediction, target) for source, target in pairs)):
                continue
            outputs = []
            valid = True
            for test in tests:
                prediction = self._recover_masked_reflection(test, mask_color)
                if prediction is None:
                    valid = False
                    break
                outputs.append(prediction)
            if valid:
                yield Candidate(f'verified_masked_reflection:c{mask_color}', outputs, 1.2)

    def solve_verified_geometry(self, task):
        programs = list(self._fit_border_ray_obstacle(task)) + list(self._fit_masked_reflection(task))
        if not programs:
            return None
        reference = programs[0].outputs
        for program in programs[1:]:
            if len(program.outputs) != len(reference):
                return None
            if any((not np.array_equal(left, right) for left, right in zip(program.outputs, reference))):
                return None
        names = ','.join((program.name for program in programs))
        return (names, reference)


In [ ]:
%%writefile geometry_merge.py
"""Apply unambiguous, demonstration-fitting geometry to the second guess only."""
import copy
import json
from pathlib import Path
import sys
import time
from verified_geometry import Arc2SymbolicSolver
from reasoning_study_policy import valid_grid,valid_answers,validate_submission

def merge(tasks,baseline,deadline):
    validate_submission(baseline,tasks)
    assert all(valid_grid(p['input']) and valid_grid(p['output']) for t in tasks.values() for p in t['train'])
    assert all(valid_grid(p['input']) and 'output' not in p for t in tasks.values() for p in t['test'])
    prediction=copy.deepcopy(baseline)
    solver=Arc2SymbolicSolver();started=time.time()
    report={'attempted_tasks':0,'verified_outputs':0,'changed_outputs':0,'deadline_stopped':False,'records':[]}
    for key,task in tasks.items():
        if time.time()>=deadline:
            report['deadline_stopped']=True;break
        report['attempted_tasks']+=1;tick=time.time()
        record={'task_id':key,'program_names':None,'changed_indices':[]}
        try:
            verified=solver.solve_verified_geometry(task)
            if verified is not None:
                names,arrays=verified;grids=[a.astype(int).tolist() for a in arrays]
                assert valid_answers(grids,task)
                record.update(program_names=names,grids=grids)
                report['verified_outputs']+=len(grids)
                for index,grid in enumerate(grids):
                    row=prediction[key][index]
                    if grid not in row.values():
                        row['attempt_2']=grid;report['changed_outputs']+=1
                        record['changed_indices'].append(index)
        except Exception as error:
            record['error']=repr(error)[:300]
        record['seconds']=time.time()-tick;report['records'].append(record)
    report['elapsed_seconds']=time.time()-started
    assert all(prediction[k][i]['attempt_1']==baseline[k][i]['attempt_1']
               for k in tasks for i in range(len(tasks[k]['test'])))
    validate_submission(prediction,tasks)
    return prediction,report

if __name__=='__main__':
    tasks=json.loads(Path(sys.argv[1]).read_text())
    baseline=json.loads(Path(sys.argv[2]).read_text())
    prediction,report=merge(tasks,baseline,float(sys.argv[4]))
    Path(sys.argv[3]).write_text(json.dumps(prediction))
    Path('geometry_report.json').write_text(json.dumps(report,indent=2))


In [ ]:
Path('LICENSE.geometry.txt').write_text('\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n')

In [ ]:
import subprocess,sys
process=subprocess.run([sys.executable,'two_pass_run.py',json.dumps(run_config)],
                       timeout=max(60,global_end_time-time.time()+120))
assert process.returncode==0,'Two-pass inference failed; inspect phase logs'


In [ ]:
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder,score_full_probmul_3
from two_pass_policy import anchored,submission_for
dm=ArcDataset.from_file(str(input_path)).split_multi_replies()
dec_a=ArcDecoder(dm,n_guesses=2)
dec_a.load_decoded_results('/kaggle/inference_outputs')
pool=ArcDecoder(dm,n_guesses=2)
pool.load_decoded_results('/kaggle/inference_outputs')
pool.load_decoded_results('/kaggle/inference_outputs_b',run_name='.b')
assert dec_a.decoded_results,'No model candidates were generated'
def plain(ranked):
    return {k:[g.tolist() for g in v] for k,v in ranked.items()}
first=plain(dec_a.run_selection_algo(score_full_probmul_3))
combined=plain(pool.run_selection_algo(score_full_probmul_3))
policies={'pass_a':submission_for(tasks,first),'pooled':submission_for(tasks,combined),
          'anchored_pool':submission_for(tasks,anchored(first,combined))}
for name,prediction in policies.items():
    Path(f'predictions_{name}.json').write_text(json.dumps(prediction))
Path('submission.json').write_text(json.dumps(policies['anchored_pool']))
report={'tasks':len(tasks),'outputs':sum(len(task['test']) for task in tasks.values()),
        'primary_covered_outputs':len(first),'pooled_covered_outputs':len(combined),
        'primary_candidates':sum(map(len,dec_a.decoded_results.values())),
        'pooled_candidates':sum(map(len,pool.decoded_results.values())),
        'notebook_seconds':time.time()-T0,'labels_opened':False,
        'submission_sha256':hashlib.sha256(Path('submission.json').read_bytes()).hexdigest()}
Path('run_report.json').write_text(json.dumps(report,indent=2))
print('Fixed predictions saved before evaluation labels:',report)


In [ ]:
# Current-input-only CPU geometry; preserve the neural result on timeout or failure.
from reasoning_study_policy import validate_submission
import subprocess,sys
geometry_deadline=min(time.time()+120,global_end_time+480)
geometry_path=Path('submission.geometry.json')
if geometry_deadline>time.time():
    with open('geometry.log','w') as logfile:
        try:
            result=subprocess.run([sys.executable,'geometry_merge.py',str(input_path),'predictions_anchored_pool.json',
                                   str(geometry_path),str(geometry_deadline)],stdout=logfile,stderr=subprocess.STDOUT,
                                   timeout=max(.1,geometry_deadline-time.time()))
            if result.returncode==0:
                prediction=json.loads(geometry_path.read_text())
                validate_submission(prediction,tasks)
                assert all(prediction[k][i]['attempt_1']==policies['anchored_pool'][k][i]['attempt_1']
                           for k in tasks for i in range(len(tasks[k]['test'])))
                policies['geometry']=prediction
            else:
                Path('geometry_failure.json').write_text(json.dumps({'returncode':result.returncode}))
        except subprocess.TimeoutExpired:
            Path('geometry_failure.json').write_text(json.dumps({'timeout_seconds':120}))
policies.setdefault('geometry',policies['anchored_pool'])
Path('predictions_geometry.json').write_text(json.dumps(policies['geometry']))
Path('submission.json').write_text(json.dumps(policies['geometry']))
report.update(notebook_seconds=time.time()-T0,submission_sha256=hashlib.sha256(Path('submission.json').read_bytes()).hexdigest(),
              geometry_policy='exact-demonstration-fit,unambiguous,replace-second-only,bounded120sec')
Path('run_report.json').write_text(json.dumps(report,indent=2))
print('Geometry prediction fixed before labels:',report)


In [ ]:
if not RERUN:
    solutions=json.loads((data_root/'arc-agi_evaluation_solutions.json').read_text())
    comparison=[]
    for name,prediction in policies.items():
        correct={(k,i): answer in prediction[k][i].values() for k,answers in solutions.items() for i,answer in enumerate(answers)}
        score=sum(sum(correct[k,i] for i in range(len(answers)))/len(answers) for k,answers in solutions.items())/len(solutions)*100
        comparison.append({'policy':name,'task_mean_percent':score,'exact_outputs':sum(correct.values())})
    Path('evaluation_report.json').write_text(json.dumps(comparison,indent=2))
    print(comparison)
